# INAR/BINAR By-Claim Reproduction

Source family: McKenzie (1985), Al-Osh and Alzaid (1987), Du
and Li (1991), Pedeli and Karlis (2011), plus the local
INAR/BINAR by-claim simulation scripts.

Paper-specific target: reproduce the deterministic expected
by-claim count recursions and terminal-reserve means used to
validate the package.

Reviewer rubric: please separate comments into mathematical
correctness, API/usability, and teaching value.

In [ ]:
import numpy as np

from ruin_theory import (
    BINARByClaimModel,
    INARByClaimModel,
    deterministic,
    simulate_binar_byclaim_terminal_reserves,
    simulate_inar_byclaim_terminal_reserves,
)

inar = INARByClaimModel(
    initial_capital=0.0,
    premium_per_period=37.0,
    primary_count_mean=10.0,
    initial_byclaim_mean=10.0,
    reproduction=0.1,
    primary_distribution=deterministic(2.0),
    byclaim_distribution=deterministic(3.0),
)

inar_counts = inar.expected_byclaim_counts(4)
inar_expected_terminal = inar.expected_terminal_reserve(4)
inar_counts, inar_expected_terminal

In [ ]:
np.testing.assert_allclose(inar_counts, [11.0, 11.1, 11.11, 11.111])
manual_terminal = 4 * 37.0 - 4 * 10.0 * 2.0 - np.sum(inar_counts) * 3.0
assert inar_expected_terminal == manual_terminal

reserves = simulate_inar_byclaim_terminal_reserves(
    inar,
    periods=8,
    n_simulations=60_000,
    seed=123,
)
assert abs(reserves.mean() - inar.expected_terminal_reserve(8)) < 0.45

In [ ]:
binar = BINARByClaimModel(
    initial_capital=1000.0,
    premium_per_period=15000.0,
    primary_count_means=(5.0, 7.0),
    initial_byclaim_means=(1.0, 1.0),
    reproduction_matrix=((0.41, 0.1), (0.05, 0.3)),
    primary_distributions=(deterministic(10.0), deterministic(1.0)),
    byclaim_distributions=(deterministic(0.5), deterministic(0.5)),
)

matrix = np.array([[0.41, 0.1], [0.05, 0.3]])
expected = np.empty((3, 2))
previous = matrix @ np.ones(2) + np.array([5.0, 7.0])
expected[0] = previous
for index in range(1, 3):
    previous = matrix @ previous + np.array([5.0, 7.0])
    expected[index] = previous

np.testing.assert_allclose(binar.expected_byclaim_counts(3), expected)
expected